# 30-Day Hospital Readmission: corrected v9 model

This notebook builds the corrected encounter features from the competition CSVs, runs five-fold validation, fits corrected LightGBM and Explainable Boosting Machine (EBM) models, then rank-blends their predictions and writes a Kaggle submission.

The fixed validation split is `StratifiedKFold(5, shuffle=True, random_state=0)` on `train.csv` row order. The expected reference scores are LightGBM **0.752053**, EBM **0.750411**, and the 0.65/0.35 rank blend about **0.7526**. These are CV results, not guaranteed leaderboard scores.

Run it in a Kaggle notebook with the competition dataset attached. The full run is CPU-heavy. The EBM stage is the slowest (about 25 minutes in the measured local environment). If EBM or `interpret-core` is unavailable, the lighter fallback is the corrected LightGBM-only submission; it is valid, but it is not the v9 blend and its CV AUC is 0.752053.

Use only the competition data. The output is `/kaggle/working/v9_corrected_lgb_ebm_blend.csv`.


In [ ]:
import glob, os, shutil, subprocess, sys, time
from pathlib import Path

# Kaggle competition data is attached as an input.
matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert matches, 'Attach the competition dataset to this Kaggle notebook.'
DATA = str(Path(matches[0]).parent)
assert Path(DATA, 'train.csv').exists(), 'Attach the competition dataset or set DATA_DIR.'
OUT = '/kaggle/working'
os.chdir(OUT)
os.environ['DATA_DIR'] = DATA
os.environ['ART_DIR'] = str(Path(OUT, 'artifacts'))
Path('artifacts').mkdir(exist_ok=True)
Path('data').mkdir(exist_ok=True)
for name in ('train.csv', 'test.csv', 'sample_submission.csv'):
    shutil.copy2(Path(DATA, name), Path('data', name))
print('competition input:', DATA)
print('working output:', OUT)
print('CPU threads:', os.cpu_count())


## 1. Check the raw split

The source tables include both train and test encounters. Feature construction applies the same row-level cleaning and aggregation to both splits. Outcome-informed transformations (target encodings, category-level alignment, feature selection, and model fitting) are learned using training-fold rows only.

In [ ]:
import pandas as pd
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
assert train.encounter_id.is_unique and test.encounter_id.is_unique
assert set(train.encounter_id).isdisjoint(test.encounter_id)
assert len(test) == len(sample) and set(test.encounter_id) == set(sample.encounter_id)
print('train:', train.shape, '| positive rate:', round(train.readmit_30d.mean(), 5))
print('test:', test.shape, '| sample submission:', sample.shape)


## 2a. Corrected base feature table (298 columns)

This stage cleans and aggregates the linked raw tables by `encounter_id`.

In [ ]:
%%writefile /kaggle/working/features_v4.py
"""Build the corrected v1 encounter table without cross-split feature counts.

Run from the repository root with the competition CSVs in data/:
    python src/features_v4.py
Produces artifacts/X4.pkl. Prior-utilization features are computed per encounter,
diagnosis vocabularies use training encounters only, and global stay-count
features based on train+test have been removed. Categorical levels are aligned
to each training fold by cv_v4.py.
"""
import pandas as pd, numpy as np
import os
D=os.environ.get('DATA_DIR','data')+'/'
ART=os.environ.get('ART_DIR','artifacts')+'/'
def rd(n,**k): return pd.read_csv(D+n+'.csv',**k)
train_ids=set(rd('train').encounter_id)

# ---------- base table: index encounters + patients + hospitals ----------
ie=rd('index_encounters',parse_dates=['admit_dt','discharge_dt'])
pt=rd('patients'); hp=rd('hospitals')
pt['insurance_type']=pt.insurance_type.str.title()          # MEDICARE -> Medicare
pt['race_ethnicity']=pt.race_ethnicity.replace('?',np.nan)
pt['sex']=pt.sex.replace('U',np.nan)
pt.loc[pt.birth_year<1910,'birth_year']=np.nan
for c in ['admission_type','admission_source','discharge_disposition']:
    ie[c]=ie[c].replace({'Not Available':np.nan,'Not Mapped':np.nan})
df=ie.merge(pt,on='patient_id',how='left').merge(hp,on='hospital_id',how='left')
df['los_days']=(df.discharge_dt-df.admit_dt).dt.total_seconds()/86400
df.loc[(df.los_days<=0)|(df.los_days>120),'los_days']=np.nan
df['age']=df.admit_dt.dt.year-df.birth_year
df['admit_hour']=df.admit_dt.dt.hour; df['admit_dow']=df.admit_dt.dt.dayofweek
df['admit_month']=df.admit_dt.dt.month; df['admit_year']=df.admit_dt.dt.year
df['weight_missing']=df.admit_weight_kg.isna().astype(int)
df['spec_missing']=df.attending_specialty.isna().astype(int)
df['zip3']=df.zip3.astype('float')

# ---------- diagnoses ----------
dx=rd('diagnoses').drop_duplicates(['encounter_id','seq'])
def fix(c):
    c=c.strip().upper()
    return c if '.' in c else (c[:3]+'.'+c[3:] if len(c)>3 else c)
dx['icd10_code']=dx.icd10_code.map(fix)
lk=rd('icd10_lookup'); dx=dx.merge(lk[['icd10_code','chapter','chronic_flag']],on='icd10_code',how='left')
dx['poa_n']=(dx.present_on_admission=='N').astype(int)
g=dx.groupby('encounter_id')
f_dx=pd.DataFrame({'dx_n':g.size(),'dx_chronic_n':g.chronic_flag.sum(),'dx_poa_n':g.poa_n.sum(),'dx_poa_missing':g.present_on_admission.apply(lambda s:s.isna().sum())})
prim=dx[dx.is_primary==1].drop_duplicates('encounter_id').set_index('encounter_id')
f_dx['dx_primary']=prim.icd10_code; f_dx['dx_primary_chapter']=prim.chapter
f_dx['dx_primary3']=prim.icd10_code.str[:3]
ch=pd.crosstab(dx.encounter_id,dx.chapter).add_prefix('dxch_'); f_dx=f_dx.join(ch)
d3=dx.assign(c3=dx.icd10_code.str[:3])
top=d3[d3.encounter_id.isin(train_ids)].c3.value_counts().head(60).index
f_dx=f_dx.join(pd.crosstab(d3[d3.c3.isin(top)].encounter_id,d3[d3.c3.isin(top)].c3).clip(upper=1).add_prefix('dx3_'))

# ---------- procedures ----------
pr=rd('procedures').drop_duplicates(); pr['code']=pr.code.astype(str)
f_pr=pr.groupby('encounter_id').agg(proc_n=('code','size'),proc_first=('code','first'))

# ---------- labs: unit harmonisation + plausibility ----------
lab=rd('lab_results').drop_duplicates()
lab['collected_dt']=pd.to_datetime(lab.collected_dt,errors='coerce')
conv={('Albumin','g/L'):.1,('Creatinine','umol/L'):1/88.4,('Glucose','mmol/L'):18.016,('Hemoglobin','g/L'):.1}
fac=[conv.get((n,u),1.) for n,u in zip(lab.lab_name,lab.unit)]
lab['v']=lab.value*np.array(fac)
rng={'Albumin':(1,6),'Bnp':(5,40000),'Bun':(2,300),'Creatinine':(.2,20),'Glucose':(30,1000),'Hba1C':(3,18),'Hemoglobin':(3,22),'Inr':(.7,15),'Lactate':(.2,30),'Potassium':(1.5,9),'Sodium':(100,180),'Troponin_I':(0,50),'Wbc':(.5,100)}
lo=lab.lab_name.map(lambda x:rng[x][0]); hi=lab.lab_name.map(lambda x:rng[x][1])
lab.loc[(lab.v<lo)|(lab.v>hi),'v']=np.nan            # unit-slip / typo outliers -> missing
lab=lab.sort_values(['encounter_id','collected_dt'])
g=lab.groupby(['encounter_id','lab_name']).v
f_lab=pd.concat({'last':g.last(),'mean':g.mean(),'min':g.min(),'max':g.max(),'first':g.first(),'cnt':g.count()},axis=1).unstack('lab_name')
f_lab.columns=[f'lab_{b}_{a}' for a,b in f_lab.columns]
for l in rng: f_lab[f'lab_{l}_delta']=f_lab[f'lab_{l}_last']-f_lab[f'lab_{l}_first']
f_lab=f_lab.drop(columns=[c for c in f_lab if c.endswith('_first')])
ab=pd.crosstab(lab.encounter_id,lab.abnormal_flag)
f_lab['lab_abn_H']=ab.get('H'); f_lab['lab_abn_L']=ab.get('L'); f_lab['lab_total']=lab.groupby('encounter_id').size()

# ---------- vitals ----------
v=rd('vitals').drop_duplicates()
v['t']=pd.to_datetime(v.charttime,format='%m/%d/%Y %H:%M',errors='coerce')
v.loc[(v.vital_type=='TEMP')&(v.value>45),'value']=(v.value-32)*5/9      # Fahrenheit -> Celsius
r={'HR':(20,250),'SBP':(50,260),'DBP':(20,160),'RR':(4,60),'SPO2':(50,100),'TEMP':(30,43)}
lo=v.vital_type.map(lambda x:r[x][0]); hi=v.vital_type.map(lambda x:r[x][1])
v.loc[(v.value<lo)|(v.value>hi),'value']=np.nan
v=v.sort_values(['encounter_id','t'])
g=v.groupby(['encounter_id','vital_type']).value
f_v=pd.concat({'last':g.last(),'first':g.first(),'mean':g.mean(),'min':g.min(),'max':g.max(),'std':g.std(),'cnt':g.count()},axis=1).unstack('vital_type')
f_v.columns=[f'vit_{b}_{a}' for a,b in f_v.columns]
for k in r: f_v[f'vit_{k}_delta']=f_v[f'vit_{k}_last']-f_v[f'vit_{k}_first']
f_v=f_v.drop(columns=[c for c in f_v if c.endswith('_first')])

# ---------- medications ----------
m=rd('medications').drop_duplicates()
m['s']=pd.to_datetime(m.start_dt.astype(str),format='%Y%m%d',errors='coerce'); m['e']=pd.to_datetime(m.stop_dt.astype('Int64').astype(str),format='%Y%m%d',errors='coerce')
g=m.groupby('encounter_id')
f_m=pd.DataFrame({'med_n':g.size(),'med_distinct':g.drug_name.nunique(),'med_cont_n':g.continued_at_discharge.sum(),'med_cont_missing':g.continued_at_discharge.apply(lambda s:s.isna().sum()),
  'med_iv_n':g.route.apply(lambda s:(s=='IV').sum()),'med_nostop_n':g.stop_dt.apply(lambda s:s.isna().sum())})
f_m=f_m.join(pd.crosstab(m.encounter_id,m.drug_class).clip(upper=1).add_prefix('med_'))

# ---------- unit transfers ----------
u=rd('unit_transfers',parse_dates=['in_dt','out_dt']).drop_duplicates()
u['h']=(u.out_dt-u.in_dt).dt.total_seconds()/3600; u.loc[(u.h<0),'h']=np.nan
f_u=u.groupby('encounter_id').agg(tr_n=('unit','size'),tr_last_unit=('unit','last'),tr_first_unit=('unit','first'))
f_u=f_u.join(u.pivot_table(index='encounter_id',columns='unit',values='h',aggfunc='sum').add_prefix('tr_hours_'))
f_u['tr_icu']=(u.assign(i=u.unit=='ICU').groupby('encounter_id').i.max()).astype(int)

# ---------- discharge planning ----------
dp=rd('discharge_planning').drop_duplicates('encounter_id').set_index('encounter_id')
dp['home_health_referral']=dp.home_health_referral.map({'Y':1,'N':0}); dp['education_done']=dp.education_done.map({'Yes':1,'No':0})
dp['dp_present']=1; dp['followup_missing']=dp.followup_scheduled_days.isna().astype(int)
dp.loc[dp.followup_scheduled_days>60,'followup_scheduled_days']=np.nan

# ---------- prior utilization (only visits before admit) ----------
pu=rd('prior_utilization',parse_dates=['visit_dt']).drop_duplicates()
pu=pu.merge(df[['encounter_id','patient_id','admit_dt']],on='patient_id'); pu['d']=(pu.admit_dt-pu.visit_dt).dt.days
pu=pu[pu.d>0]
rows={}
for t in ['ED','IP','OP']:
    s=pu[pu.visit_type==t]
    for w in [30,90,180,365]: rows[f'pu_{t}_{w}d']=s[s.d<=w].groupby('encounter_id').size()
    rows[f'pu_{t}_all']=s.groupby('encounter_id').size(); rows[f'pu_{t}_lastgap']=s.groupby('encounter_id').d.min()
rows['pu_all']=pu.groupby('encounter_id').size(); rows['pu_nhosp']=pu.groupby('encounter_id').hospital_id.nunique()
f_pu=pd.DataFrame(rows)

# ---------- assemble ----------
X=df.set_index('encounter_id')
for f in [f_dx,f_pr,f_lab,f_v,f_m,f_u,dp,f_pu]: X=X.join(f)
cnt=[c for c in X if c.startswith(('pu_','dx_n','proc_n','med_n')) and 'lastgap' not in c]
for c in [c for c in X if c.startswith(('dxch_','dx3_','med_')) or c in('proc_n','pu_all')]: X[c]=X[c].fillna(0)
for c in [c for c in X if c.startswith('pu_') and 'lastgap' not in c]: X[c]=X[c].fillna(0)
X['dp_present']=X.dp_present.fillna(0)
X['pu_acute_365']=X.pu_ED_365d+X.pu_IP_365d
X['pu_acute_90']=X.pu_ED_90d+X.pu_IP_90d
X=X.drop(columns=['patient_id','admit_dt','discharge_dt','birth_year','dx_primary'],errors='ignore')
cats=[c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c]) and not pd.api.types.is_datetime64_any_dtype(X[c])]
for c in cats: X[c]=X[c].astype('category')
os.makedirs(ART,exist_ok=True); X=X.copy(); X.to_pickle(ART+'X4.pkl')
print(X.shape, len(cats),'categoricals'); print(cats)


In [ ]:
!python /kaggle/working/features_v4.py

## 2b. Add training-vocabulary diagnosis and clinical score features (316 columns)

This stage cleans and aggregates the linked raw tables by `encounter_id`.

In [ ]:
%%writefile /kaggle/working/features_v4_v2.py
"""Add v2 features to corrected X4.pkl, with diagnosis vocabulary from train only.

Run from the repository root after src/features_v4.py:
    python src/features_v4_v2.py
Produces artifacts/X4_2.pkl.
"""
import os
import numpy as np
import pandas as pd

D = os.environ.get('DATA_DIR', 'data') + '/'
ART = os.environ.get('ART_DIR', 'artifacts') + '/'
X = pd.read_pickle(ART + 'X4.pkl')
train_ids = set(pd.read_csv(D + 'train.csv').encounter_id)

# ---------- diagnoses: Charlson-style score + extra 3-char flags ----------
dx = pd.read_csv(D + 'diagnoses.csv').drop_duplicates(['encounter_id', 'seq'])
dx['icd'] = dx.icd10_code.str.strip().str.upper()
dx['c3'] = dx.icd.str[:3]

charlson = {  # prefix -> weight (simplified Charlson)
    'I21': 1, 'I22': 1, 'I50': 1, 'I70': 1, 'I73': 1, 'G45': 1, 'F00': 1, 'F01': 1, 'F02': 1, 'F03': 1,
    'G30': 1, 'J40': 1, 'J41': 1, 'J42': 1, 'J43': 1, 'J44': 1, 'J45': 1, 'J47': 1, 'M05': 1, 'M06': 1,
    'M32': 1, 'K25': 1, 'K26': 1, 'K27': 1, 'K28': 1, 'K70': 1, 'K74': 1, 'E10': 1, 'E11': 1,
    'G81': 2, 'G82': 2, 'N18': 2, 'N19': 2, 'B20': 6}
w = dx.c3.map(charlson).fillna(0)
w[dx.c3.str[0].eq('C') & dx.c3.between('C00', 'C76')] = 2
w[dx.c3.between('C77', 'C80')] = 6
dx['w'] = w
ch = dx.groupby('encounter_id').w.sum().rename('charlson')

# flags for the 150 most common 3-char codes (v1 had 60)
top = dx[dx.encounter_id.isin(train_ids)].c3.value_counts().head(150).index
fl = pd.crosstab(dx[dx.c3.isin(top)].encounter_id, dx[dx.c3.isin(top)].c3).clip(upper=1).add_prefix('dx3b_')
# only add the ones v1 did not have
fl = fl[[c for c in fl if 'dx3_' + c[5:] not in X.columns]]

# raw-code count of distinct 3-char codes and share of chronic
nd = dx.groupby('encounter_id').c3.nunique().rename('dx_distinct3')

X = X.join(ch).join(nd).join(fl)
X['charlson'] = X.charlson.fillna(0)
for c in fl.columns:
    X[c] = X[c].fillna(0)

# ---------- LACE-style score ----------
los = X.los_days
L = pd.cut(los, [-1, 1, 2, 3, 6, 13, 1000], labels=[0, 1, 2, 3, 4, 5], right=False).astype(float)
L = np.select([los < 1, los < 2, los < 3, los < 4, los < 7, los < 14], [0, 1, 2, 3, 4, 5], default=7)
A = (X.admission_type.astype(str) == 'Emergency').astype(int) * 3
C = np.select([X.charlson == 0, X.charlson == 1, X.charlson == 2, X.charlson == 3], [0, 1, 2, 3], default=5)
E = np.minimum(X.pu_ED_180d.fillna(0), 4)
X['lace'] = L + A + C + E

# ---------- ratios / interactions ----------
X['bun_creat'] = X.lab_Bun_last / X.lab_Creatinine_last
X['med_cont_frac'] = X.med_cont_n / X.med_n.replace(0, np.nan)
X['med_per_day'] = X.med_n / X.los_days
X['dx_per_day'] = X.dx_n / X.los_days
X['pu_acute_frac'] = X.pu_acute_365 / X.pu_all.replace(0, np.nan)
X['prior_ip_recent'] = (X.pu_IP_lastgap <= 30).astype(int)
X['age_x_charlson'] = X.age * X.charlson
X['los_x_charlson'] = X.los_days * X.charlson
X['icu_hours'] = X.get('tr_hours_ICU')
X['shock_index'] = X.vit_HR_last / X.vit_SBP_last

# ---------- outlier flags: how many raw lab values fell outside plausible range ----------
lab = pd.read_csv(D + 'lab_results.csv').drop_duplicates()
conv = {('Albumin', 'g/L'): .1, ('Creatinine', 'umol/L'): 1 / 88.4, ('Glucose', 'mmol/L'): 18.016, ('Hemoglobin', 'g/L'): .1}
lab['v'] = lab.value * np.array([conv.get((n, u), 1.) for n, u in zip(lab.lab_name, lab.unit)])
rng = {'Albumin': (1, 6), 'Bnp': (5, 40000), 'Bun': (2, 300), 'Creatinine': (.2, 20), 'Glucose': (30, 1000), 'Hba1C': (3, 18),
       'Hemoglobin': (3, 22), 'Inr': (.7, 15), 'Lactate': (.2, 30), 'Potassium': (1.5, 9), 'Sodium': (100, 180),
       'Troponin_I': (0, 50), 'Wbc': (.5, 100)}
lo = lab.lab_name.map(lambda x: rng[x][0]); hi = lab.lab_name.map(lambda x: rng[x][1])
lab['out'] = ((lab.v < lo) | (lab.v > hi)).astype(int)
X = X.join(lab.groupby('encounter_id').out.sum().rename('lab_outlier_n'))
X['lab_outlier_n'] = X.lab_outlier_n.fillna(0)
X = X.join(lab.groupby('encounter_id').lab_name.nunique().rename('lab_distinct_tests'))

X = X.copy()
X.to_pickle(ART + 'X4_2.pkl')
print(X.shape)
print(X[['charlson', 'lace']].describe().T)


In [ ]:
!python /kaggle/working/features_v4_v2.py

## 2c. Add discharge-state and timing features (461 stored columns)

This stage cleans and aggregates the linked raw tables by `encounter_id`.

In [ ]:
%%writefile /kaggle/working/features_v4_v3.py
"""Add v3 discharge-state features to the corrected X4_2.pkl table.
Run from repository root after src/features_v4_v2.py:
    python src/features_v4_v3.py
Produces artifacts/X4_3.pkl. Drug vocabularies use training encounters only;
same-hospital prior-use features are calculated per encounter.

  l3_*  labs:        hours between last lab and discharge, last-24h counts, abnormal flag of the last value, per-test slope
  v3_*  vitals:      last-24h mean/min/max per vital, clinical-threshold abnormal counts near discharge
  m3_*  medications: drug-name flags (names normalised: 122 spellings -> 52), started/stopped timing, high-risk drugs
  p3_*  procedures:  timing relative to admission/discharge (CPT and CPT-4 treated as one system)
  t3_*  transfers:   timing of the last unit move and of the last ICU stay relative to discharge
  u3_*  prior visits: split into same hospital as the index stay vs other hospitals
"""
import json
import os
import re
import time

import numpy as np
import pandas as pd

D = os.environ.get('DATA_DIR', 'data') + '/'
ART = os.environ.get('ART_DIR', 'artifacts') + '/'
t0 = time.time()


def log(msg):
    print(f'[{round(time.time() - t0)}s] {msg}', flush=True)


def clean(s):
    return re.sub(r'[^0-9a-zA-Z]+', '_', str(s).strip().lower()).strip('_')


X = pd.read_pickle(ART + 'X4_2.pkl')
train_ids = set(pd.read_csv(D + 'train.csv').encounter_id)
ie = pd.read_csv(D + 'index_encounters.csv', parse_dates=['admit_dt', 'discharge_dt'])[
    ['encounter_id', 'patient_id', 'hospital_id', 'admit_dt', 'discharge_dt']]
NEW = {}  # group prefix -> list of DataFrames


def add(prefix, df):
    NEW.setdefault(prefix, []).append(df)


# ================================================================ labs
lab = pd.read_csv(D + 'lab_results.csv').drop_duplicates()
lab['collected_dt'] = pd.to_datetime(lab.collected_dt, errors='coerce')
conv = {('Albumin', 'g/L'): .1, ('Creatinine', 'umol/L'): 1 / 88.4, ('Glucose', 'mmol/L'): 18.016, ('Hemoglobin', 'g/L'): .1}
lab['v'] = lab.value * np.array([conv.get((n, u), 1.) for n, u in zip(lab.lab_name, lab.unit)])
rng = {'Albumin': (1, 6), 'Bnp': (5, 40000), 'Bun': (2, 300), 'Creatinine': (.2, 20), 'Glucose': (30, 1000), 'Hba1C': (3, 18),
       'Hemoglobin': (3, 22), 'Inr': (.7, 15), 'Lactate': (.2, 30), 'Potassium': (1.5, 9), 'Sodium': (100, 180),
       'Troponin_I': (0, 50), 'Wbc': (.5, 100)}
lo = lab.lab_name.map(lambda x: rng[x][0]); hi = lab.lab_name.map(lambda x: rng[x][1])
lab.loc[(lab.v < lo) | (lab.v > hi), 'v'] = np.nan
lab = lab.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
lab['hb'] = (lab.discharge_dt - lab.collected_dt).dt.total_seconds() / 3600       # hours before discharge
lab['ta'] = (lab.collected_dt - lab.admit_dt).dt.total_seconds() / 3600          # hours since admission
lab['l24'] = (lab.hb <= 24).astype(int)
lab['abn'] = lab.abnormal_flag.isin(['H', 'L']).astype(int)
g = lab.groupby('encounter_id')
f = pd.DataFrame({'l3_hrs_last_lab': g.hb.min(), 'l3_hrs_mean_lab': g.hb.mean(), 'l3_n_last24': g.l24.sum(),
                  'l3_frac_last24': g.l24.mean(), 'l3_abn_last24': (lab.abn * lab.l24).groupby(lab.encounter_id).sum()})
add('l3', f)
lab = lab.sort_values(['encounter_id', 'lab_name', 'collected_dt'])
gl = lab.groupby(['encounter_id', 'lab_name'])
f = gl.hb.min().unstack(); f.columns = [f'l3_{clean(c)}_hrs_since' for c in f.columns]; add('l3', f)
flag = lab.assign(fl=lab.abnormal_flag.map({'H': 1, 'L': -1, 'N': 0})).groupby(['encounter_id', 'lab_name']).fl.last().unstack()
flag.columns = [f'l3_{clean(c)}_lastflag' for c in flag.columns]; add('l3', flag)
lv = lab.dropna(subset=['v']).copy(); lv['tt'] = lv.ta ** 2; lv['tv'] = lv.ta * lv.v
s = lv.groupby(['encounter_id', 'lab_name']).agg(n=('v', 'size'), st=('ta', 'sum'), sv=('v', 'sum'), stt=('tt', 'sum'), stv=('tv', 'sum'))
den = s.n * s.stt - s.st ** 2
slope = ((s.n * s.stv - s.st * s.sv) / den).where((s.n >= 3) & (den > 1e-6))
slope = slope.unstack(); slope.columns = [f'l3_{clean(c)}_slope' for c in slope.columns]; add('l3', slope)
del lab, lv, s
log('labs done')

# ================================================================ vitals
v = pd.read_csv(D + 'vitals.csv').drop_duplicates()
v['t'] = pd.to_datetime(v.charttime, format='%m/%d/%Y %H:%M', errors='coerce')
v.loc[(v.vital_type == 'TEMP') & (v.value > 45), 'value'] = (v.value - 32) * 5 / 9
r = {'HR': (20, 250), 'SBP': (50, 260), 'DBP': (20, 160), 'RR': (4, 60), 'SPO2': (50, 100), 'TEMP': (30, 43)}
lo = v.vital_type.map(lambda x: r[x][0]); hi = v.vital_type.map(lambda x: r[x][1])
v.loc[(v.value < lo) | (v.value > hi), 'value'] = np.nan
v = v.merge(ie[['encounter_id', 'discharge_dt']], on='encounter_id')
v['hb'] = (v.discharge_dt - v.t).dt.total_seconds() / 3600
x, vt = v.value, v.vital_type
v['abn'] = (((vt == 'HR') & ((x > 100) | (x < 50))) | ((vt == 'SBP') & ((x < 90) | (x > 180))) | ((vt == 'SPO2') & (x < 92))
            | ((vt == 'TEMP') & (x > 38)) | ((vt == 'RR') & (x > 24))).astype(int)
v['l24'] = (v.hb <= 24).astype(int); v['l12'] = (v.hb <= 12).astype(int)
g = v.groupby('encounter_id')
f = pd.DataFrame({'v3_hrs_last_vital': g.hb.min(), 'v3_abn_total': g.abn.sum(), 'v3_abn_frac': g.abn.mean(), 'v3_n_last24': g.l24.sum(),
                  'v3_abn_last24': (v.abn * v.l24).groupby(v.encounter_id).sum(), 'v3_abn_last12': (v.abn * v.l12).groupby(v.encounter_id).sum()})
f['v3_abn_last24_frac'] = f.v3_abn_last24 / f.v3_n_last24.replace(0, np.nan); add('v3', f)
w = v[v.l24 == 1].groupby(['encounter_id', 'vital_type']).value.agg(['mean', 'min', 'max']).unstack()
w.columns = [f'v3_{b}_l24_{a}' for a, b in w.columns]; add('v3', w)
del v
log('vitals done')

# ================================================================ medications
m = pd.read_csv(D + 'medications.csv')
m['dn'] = m.drug_name.map(clean)
m = m.drop_duplicates(['encounter_id', 'dn', 'start_dt', 'route'])
m['s'] = pd.to_datetime(m.start_dt.astype(str), format='%Y%m%d', errors='coerce')
m['e'] = pd.to_datetime(m.stop_dt.astype('Int64').astype(str), format='%Y%m%d', errors='coerce')
m = m.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
dcd, adm = m.discharge_dt.dt.normalize(), m.admit_dt.dt.normalize()
m['st_last1'] = (m.s >= dcd - pd.Timedelta(days=1)).astype(int)
m['st_adm'] = (m.s <= adm).astype(int)
m['stopped'] = (m.e < dcd).astype(int)
m['dur'] = (m.e - m.s).dt.days
hr = m.dn.str.contains('enoxaparin|heparin|warfarin|insulin|morphine|oxycodone|hydromorphone|fentanyl|digoxin|tramadol', regex=True)
m['hr'] = hr.astype(int)
g = m.groupby('encounter_id')
f = pd.DataFrame({'m3_started_last1d': g.st_last1.sum(), 'm3_started_admit_day': g.st_adm.sum(), 'm3_stopped_before_dc': g.stopped.sum(),
                  'm3_mean_dur_days': g.dur.mean(), 'm3_n_highrisk': g.hr.sum(), 'm3_n_names': g.dn.nunique()}); add('m3', f)
top = m[m.encounter_id.isin(train_ids)].dn.value_counts().head(50).index
fl = pd.crosstab(m[m.dn.isin(top)].encounter_id, m[m.dn.isin(top)].dn).clip(upper=1); fl.columns = [f'm3_drug_{c}' for c in fl.columns]; add('m3', fl)
cont = m[m.continued_at_discharge == 1].groupby('encounter_id').dn.nunique().rename('m3_n_cont_names'); add('m3', cont.to_frame())
del m
log('meds done')

# ================================================================ procedures
p = pd.read_csv(D + 'procedures.csv').drop_duplicates()
p['d'] = pd.to_datetime(p.proc_dt, format='%d-%b-%Y', errors='coerce')
p = p.merge(ie[['encounter_id', 'admit_dt', 'discharge_dt']], on='encounter_id')
p['da'] = (p.d - p.admit_dt.dt.normalize()).dt.days; p['db'] = (p.discharge_dt.dt.normalize() - p.d).dt.days
p['code'] = p.code.astype(str).str.strip()
g = p.groupby('encounter_id')
f = pd.DataFrame({'p3_first_day': g.da.min(), 'p3_last_day': g.da.max(), 'p3_last_days_before_dc': g.db.min(),
                  'p3_any_last48h': (p.db <= 2).groupby(p.encounter_id).max().astype(int), 'p3_distinct_codes': g.code.nunique()}); add('p3', f)
del p
log('procedures done')

# ================================================================ unit transfers
u = pd.read_csv(D + 'unit_transfers.csv', parse_dates=['in_dt', 'out_dt']).drop_duplicates()
u = u.merge(ie[['encounter_id', 'discharge_dt']], on='encounter_id').sort_values(['encounter_id', 'in_dt'])
u['in_hb'] = (u.discharge_dt - u.in_dt).dt.total_seconds() / 3600; u['out_hb'] = (u.discharge_dt - u.out_dt).dt.total_seconds() / 3600
u['dur'] = (u.out_dt - u.in_dt).dt.total_seconds() / 3600
icu = u.unit.astype(str).str.upper().str.contains('ICU')
g = u.groupby('encounter_id')
f = pd.DataFrame({'t3_last_in_hrs_before_dc': g.in_hb.min(), 'm_tmp': g.dur.last(), 'tmp_n24': (u.in_hb <= 24).groupby(u.encounter_id).sum()})
f = f.rename(columns={'m_tmp': 't3_last_unit_hours', 'tmp_n24': 't3_n_moves_last24'})
f['t3_hrs_since_icu_out'] = u[icu].groupby('encounter_id').out_hb.min()
f['t3_icu_last24'] = ((u.out_hb <= 24) & icu).groupby(u.encounter_id).max().astype(int); add('t3', f)
del u
log('transfers done')

# ================================================================ prior visits: same hospital vs elsewhere
pu = pd.read_csv(D + 'prior_utilization.csv', parse_dates=['visit_dt']).drop_duplicates()
pu = pu.merge(ie[['encounter_id', 'patient_id', 'hospital_id', 'admit_dt']].rename(columns={'hospital_id': 'idx_h'}), on='patient_id')
pu['d'] = (pu.admit_dt - pu.visit_dt).dt.days; pu = pu[pu.d > 0]
pu['same'] = (pu.hospital_id == pu.idx_h).astype(int)
cols = {}
for tag, mask in [('same', pu.same == 1), ('diff', pu.same == 0)]:
    for w in [90, 365, 10 ** 6]:
        s_ = pu[mask & (pu.d <= w)].groupby('encounter_id').size(); cols[f'u3_{tag}_{"all" if w > 1000 else w}d'] = s_
    cols[f'u3_{tag}_ip_all'] = pu[mask & (pu.visit_type == 'IP')].groupby('encounter_id').size()
f = pd.DataFrame(cols).fillna(0)
f['u3_same_frac'] = f.u3_same_alld / (f.u3_same_alld + f.u3_diff_alld).replace(0, np.nan)
f = f.rename(columns={'u3_same_alld': 'u3_same_all', 'u3_diff_alld': 'u3_diff_all'})
log('prior done')

# ================================================================ assemble
X3 = X.copy(); groups = {}
for pre, dfs in NEW.items():
    d = pd.concat(dfs, axis=1)
    X3 = X3.join(d); groups[pre] = list(d.columns)
X3 = X3.join(f)
groups['u3'] = list(f.columns)
zero = [c for c in X3.columns if c.startswith(('u3_', 'm3_drug_', 'm3_n', 'm3_started', 'm3_stopped', 'v3_n_', 'v3_abn', 'l3_n_', 'l3_abn_', 't3_n_', 't3_icu_', 'p3_any'))]
for c in zero:
    if c in X3 and c != 'u3_same_frac':
        X3[c] = X3[c].fillna(0)
X3 = X3.copy(); X3.to_pickle(ART + 'X4_3.pkl'); json.dump(groups, open(ART + 'v4_groups.json', 'w'))
log(f'X4_3 {X3.shape} (X4_2 had {X.shape[1]} cols); new per group: ' + str({k: len(v) for k, v in groups.items()}))
print(X3[[c for c in X3 if c[:3] in ('l3_', 'v3_', 'm3_', 'p3_', 't3_', 'u3_')]].isna().mean().round(2).describe())


In [ ]:
!python /kaggle/working/features_v4_v3.py

## 3. Install the EBM dependency if needed

LightGBM and scikit-learn are usually available in Kaggle. The EBM comes from `interpret-core`. If it is missing, this cell attempts to install it; internet must be enabled for that one install. If installation is unavailable, skip the EBM cells and use the corrected LightGBM-only fallback described above.

In [ ]:
try:
    import interpret.glassbox
    print('interpret-core is available')
except ImportError:
    print('Installing interpret-core for the EBM stage. This requires Kaggle Internet access.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'interpret-core==0.7.8'])


### Fold-local target encoding helpers

In [ ]:
%%writefile /kaggle/working/cv_v2.py
import sys
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

TE_COLS = ['hospital_id', 'attending_id', 'dx_primary3', 'proc_first']
M = 50  # smoothing strength


def te_fit(col, y):
    """category -> smoothed mean of y, plus global mean"""
    g = y.mean()
    s = pd.DataFrame({'c': col.astype(object).values, 'y': y}).groupby('c').y.agg(['sum', 'count'])
    return ((s['sum'] + M * g) / (s['count'] + M)), g


def te_apply(col, mp):
    m, g = mp
    return col.astype(object).map(m).astype(float).fillna(g).values


def add_te(Xtr, ytr, Xva_list, seed=0):
    """OOF target encoding for the train part, full-train encoding for the others."""
    Xtr = Xtr.copy(); outs = [x.copy() for x in Xva_list]
    for c in TE_COLS:
        oof = np.zeros(len(Xtr))
        for a, b in StratifiedKFold(5, shuffle=True, random_state=seed).split(Xtr, ytr):
            oof[b] = te_apply(Xtr[c].iloc[b], te_fit(Xtr[c].iloc[a], ytr[a]))
        mp = te_fit(Xtr[c], ytr)
        Xtr['te_' + c] = oof
        for o in outs:
            o['te_' + c] = te_apply(o[c], mp)
    return Xtr, outs


P = dict(objective='binary', learning_rate=0.03, num_leaves=15, min_child_samples=50, feature_fraction=0.5,
         bagging_fraction=0.8, bagging_freq=1, lambda_l2=5, cat_smooth=50, min_data_per_group=100, verbose=-1, n_jobs=-1)


def prep(X):
    # attending_id and proc_first are high-cardinality: drop raw, keep target-encoded version
    return X.drop(columns=['attending_id', 'proc_first'], errors='ignore')


def run(path, use_te, name, over=None):
    P.update(over or {})
    X = pd.read_pickle(path)
    tr = pd.read_csv('data/train.csv'); y = tr.readmit_30d.values
    Xt = X.loc[tr.encounter_id]
    oof = np.zeros(len(y)); its = []
    for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(Xt, y):
        A, B = Xt.iloc[a], Xt.iloc[b]
        if use_te:
            A, (B,) = add_te(A, y[a], [B])
        A, B = prep(A), prep(B)
        m = lgb.train(P, lgb.Dataset(A, y[a]), 3000, valid_sets=[lgb.Dataset(B, y[b])],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = m.predict(B); its.append(m.best_iteration)
        print(name, 'fold', round(roc_auc_score(y[b], oof[b]), 4), m.best_iteration, flush=True)
    print(name, 'OOF AUC', round(roc_auc_score(y, oof), 4), 'mean iters', int(np.mean(its)), flush=True)
    np.save(f'artifacts/oof_{name}.npy', oof)


if __name__ == '__main__':
    import json
    run(sys.argv[1], sys.argv[2] == '1', sys.argv[3], json.loads(sys.argv[4]) if len(sys.argv) > 4 else None)


### Corrected LightGBM cross-validation and test fit

In [ ]:
%%writefile /kaggle/working/lgbm_corrected_v2.py
"""Corrected LightGBM with provider/hospital counts fit inside each fold.

Run from the repository root after:
    python src/features_v4.py
    python src/features_v4_v2.py
    python src/features_v4_v3.py
Target encodings and categorical levels are fitted within each fold. Outputs
artifacts/oof_lgbm_corrected_v2.npy and artifacts/test_lgbm_corrected_v2.npy.
"""
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from cv_v2 import add_te, prep, P as BASE_P


def align_categories(train, *others):
    """Use levels observed in this training partition; unknowns become missing."""
    train = train.copy()
    others = [frame.copy() for frame in others]
    cat_cols = [c for c in train.columns
                if str(train[c].dtype) == 'category' or not pd.api.types.is_numeric_dtype(train[c])]
    for col in cat_cols:
        levels = pd.Index(train[col].dropna().astype(object).unique())
        train[col] = pd.Categorical(train[col].astype(object), categories=levels)
        for frame in others:
            frame[col] = pd.Categorical(frame[col].astype(object), categories=levels)
    return (train, *others)


def add_fold_counts(train, *others):
    """Map counts learned from train-fold encounters to train/validation/test."""
    train = train.copy()
    others = [frame.copy() for frame in others]
    for col in ('attending_id', 'hospital_id'):
        counts = train[col].astype(object).value_counts(dropna=True)
        train[col + '_n_fold'] = train[col].astype(object).map(counts).fillna(0).astype(float)
        for frame in others:
            frame[col + '_n_fold'] = frame[col].astype(object).map(counts).fillna(0).astype(float)
    return (train, *others)


def main():
    X = pd.read_pickle('artifacts/X4_3.pkl')
    tr, te = pd.read_csv('data/train.csv'), pd.read_csv('data/test.csv')
    y = tr.readmit_30d.to_numpy()
    Xtr, Xte = X.loc[tr.encounter_id], X.loc[te.encounter_id]
    params = dict(BASE_P, num_leaves=4, min_child_samples=400,
                  feature_fraction=0.2, lambda_l2=60, learning_rate=0.04)
    oof = np.zeros(len(y))
    rounds = []
    for fold, (a, b) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(Xtr, y)):
        A, B = add_fold_counts(Xtr.iloc[a], Xtr.iloc[b])
        A, (B,) = add_te(A, y[a], [B], seed=fold)
        A, B = prep(A), prep(B)
        A, B = align_categories(A, B)
        model = lgb.train(params, lgb.Dataset(A, y[a]), 3000,
                          valid_sets=[lgb.Dataset(B, y[b])],
                          callbacks=[lgb.early_stopping(100, verbose=False)])
        oof[b] = model.predict(B)
        rounds.append(model.best_iteration)
        print('fold', fold, 'AUC', round(roc_auc_score(y[b], oof[b]), 5),
              'rounds', model.best_iteration, flush=True)

    A, B = add_fold_counts(Xtr, Xte)
    A, (B,) = add_te(A, y, [B], seed=0)
    A, B = prep(A), prep(B)
    A, B = align_categories(A, B)
    final = lgb.train(params, lgb.Dataset(A, y), max(1, int(np.mean(rounds) * 1.1)))
    test_pred = final.predict(B)
    auc = roc_auc_score(y, oof)
    np.save('artifacts/oof_lgbm_corrected_v2.npy', oof)
    np.save('artifacts/test_lgbm_corrected_v2.npy', test_pred)
    print('rows/features', X.shape, 'OOF AUC', round(auc, 6),
          'test mean', round(float(test_pred.mean()), 6), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
!python /kaggle/working/lgbm_corrected_v2.py

### Corrected EBM cross-validation and test fit

In [ ]:
%%writefile /kaggle/working/ebm_top150_corrected_v1.py
"""EBM on corrected features with fold-local LightGBM selection and categories.

Run from the repository root after features_v2.py has created artifacts/X2.pkl:
    python src/ebm_top150_corrected_v1.py
Each fold selects top 150 features and category levels using that fold's training
rows. Writes OOF and fold-ensemble test probabilities under artifacts/.
"""
import numpy as np
import pandas as pd
import lightgbm as lgb
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import cv_v2 as c
from lgbm_corrected_v2 import add_fold_counts, align_categories


def main():
    X = pd.read_pickle('artifacts/X4_3.pkl')
    tr, te = pd.read_csv('data/train.csv'), pd.read_csv('data/test.csv')
    y = tr.readmit_30d.to_numpy()
    Xtr, Xte = X.loc[tr.encounter_id], X.loc[te.encounter_id]
    params = dict(c.P, num_leaves=4, min_child_samples=400,
                  feature_fraction=0.2, lambda_l2=60, learning_rate=0.04)
    folds = StratifiedKFold(5, shuffle=True, random_state=0).split(Xtr, y)
    oof, test_pred = np.zeros(len(y)), np.zeros(len(te))

    for fold, (a, b) in enumerate(folds):
        A, B = add_fold_counts(Xtr.iloc[a], Xtr.iloc[b])
        A, (B,) = c.add_te(A, y[a], [B], seed=fold)
        A, T = add_fold_counts(Xtr.iloc[a], Xte)
        A, (T,) = c.add_te(A, y[a], [T], seed=fold)
        A, B, T = c.prep(A), c.prep(B), c.prep(T)
        A, B, T = align_categories(A, B, T)
        selector = lgb.train(dict(params, seed=0), lgb.Dataset(A, y[a]), 700)
        gain = selector.feature_importance(importance_type='gain')
        chosen = np.argsort(gain)[-min(150, len(gain)):]
        names = A.columns[chosen].tolist()
        ebm_A, ebm_B, ebm_T = A[names].copy(), B[names].copy(), T[names].copy()
        cats = [col for col in names if str(ebm_A[col].dtype) == 'category']
        for col in cats:
            levels = pd.Index(ebm_A[col].dropna().astype(object).unique())
            ebm_A[col] = ebm_A[col].astype(object).where(ebm_A[col].notna(), 'missing')
            ebm_B[col] = pd.Categorical(ebm_B[col].astype(object).where(ebm_B[col].notna(), 'missing'),
                                        categories=list(levels) + ['missing'])
            ebm_T[col] = pd.Categorical(ebm_T[col].astype(object).where(ebm_T[col].notna(), 'missing'),
                                        categories=list(levels) + ['missing'])
        ebm_A = ebm_A.replace([np.inf, -np.inf], np.nan)
        ebm_B = ebm_B.replace([np.inf, -np.inf], np.nan)
        ebm_T = ebm_T.replace([np.inf, -np.inf], np.nan)
        model = ExplainableBoostingClassifier(
            max_bins=64, interactions=10, outer_bags=2, inner_bags=0,
            learning_rate=0.015, max_rounds=2000, early_stopping_rounds=100,
            min_samples_leaf=50, n_jobs=-1, random_state=fold)
        model.fit(ebm_A, y[a])
        oof[b] = model.predict_proba(ebm_B)[:, 1]
        test_pred += model.predict_proba(ebm_T)[:, 1] / 5
        print('fold', fold, 'AUC', round(roc_auc_score(y[b], oof[b]), 5),
              'selected', len(names), flush=True)

    auc = roc_auc_score(y, oof)
    np.save('artifacts/oof_ebm_top150_corrected_v1.npy', oof)
    np.save('artifacts/test_ebm_top150_corrected_v1.npy', test_pred)
    print('OOF AUC', round(auc, 6), 'test mean', round(float(test_pred.mean()), 6), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
!python /kaggle/working/ebm_top150_corrected_v1.py

### OOF evaluation and probability-scale rank blend

In [ ]:
%%writefile /kaggle/working/blend_corrected_lgb_ebm_v1.py
"""Build a corrected-feature LightGBM/EBM rank blend for offline evaluation.

Run after lgbm_corrected_v2.py and ebm_top150_corrected_v1.py:
    python src/blend_corrected_lgb_ebm_v1.py
Uses fixed 0.65/0.35 ranks (selected on an 80% OOF split), reports the held-out
20%, and rescales test ranks to corrected LightGBM probabilities. No CSV is made.
"""
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


def main():
    y = pd.read_csv('data/train.csv').readmit_30d.to_numpy()
    oof_lgb = np.load('artifacts/oof_lgbm_corrected_v2.npy')
    oof_ebm = np.load('artifacts/oof_ebm_top150_corrected_v1.npy')
    rank_lgb, rank_ebm = rankdata(oof_lgb) / len(y), rankdata(oof_ebm) / len(y)
    oof = 0.65 * rank_lgb + 0.35 * rank_ebm
    a, b = train_test_split(np.arange(len(y)), test_size=0.2,
                            random_state=0, stratify=y)
    print('OOF AUC', round(roc_auc_score(y, oof), 6),
          'weight-fit split AUC', round(roc_auc_score(y[a], oof[a]), 6),
          'holdout AUC', round(roc_auc_score(y[b], oof[b]), 6), flush=True)

    test_lgb = np.load('artifacts/test_lgbm_corrected_v2.npy')
    test_ebm = np.load('artifacts/test_ebm_top150_corrected_v1.npy')
    blend_test = 0.65 * rankdata(test_lgb) / len(test_lgb) + 0.35 * rankdata(test_ebm) / len(test_ebm)
    order = rankdata(blend_test, method='ordinal').astype(int) - 1
    probabilities = np.sort(test_lgb)[order]
    np.save('artifacts/oof_blend_corrected_lgb_ebm_v1.npy', oof)
    np.save('artifacts/test_blend_corrected_lgb_ebm_v1.npy', probabilities)
    print('test mean', round(float(probabilities.mean()), 6),
          'range', (float(probabilities.min()), float(probabilities.max())), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
!python /kaggle/working/blend_corrected_lgb_ebm_v1.py

## 4. Write and check the submission

The rank blend is converted back to a probability scale by sorting the corrected LightGBM test probabilities and assigning them in blended-rank order. This preserves a probability value for every test encounter.

In [ ]:
import numpy as np
blend_path = Path('artifacts/test_blend_corrected_lgb_ebm_v1.npy')
if blend_path.exists():
    pred = np.load(blend_path)
    print('Using the corrected LightGBM + EBM v9 blend.')
else:
    pred = np.load('artifacts/test_lgbm_corrected_v2.npy')
    print('EBM was skipped; using the corrected LightGBM-only fallback, not the v9 blend.')
test = pd.read_csv('data/test.csv')
sample = pd.read_csv('data/sample_submission.csv')
submission = pd.DataFrame({'encounter_id': test.encounter_id, 'readmit_30d': pred})
assert list(submission.columns) == ['encounter_id', 'readmit_30d']
assert len(submission) == len(sample) and submission.encounter_id.is_unique
assert set(submission.encounter_id) == set(sample.encounter_id)
assert submission.readmit_30d.notna().all() and submission.readmit_30d.between(0, 1).all()
path = '/kaggle/working/v9_corrected_lgb_ebm_blend.csv'
submission.to_csv(path, index=False)
print('saved:', path, '| shape:', submission.shape,
      '| probability range:', (submission.readmit_30d.min(), submission.readmit_30d.max()))
